# 🤗 Day 1 — EXAONE 모델 로드 & 첫 번째 추론 호출

---

## 🎯 오늘의 상황

> 로컬 LLM으로 나만의 AI 서비스를 만들어 보고 싶은데, 어디서부터 시작해야 할까요?  
> API 모델과 달리 로컬 모델은 **"열쇠"가 필요 없습니다.** 대신 **내 GPU에 모델 자체를 직접 불러와야** 합니다.  
> 그게 바로 **모델 로드(Model Load)** 입니다.
>
> 오늘은 EXAONE 모델을 불러오고, **처음으로 추론을 요청해봅니다.**

---

## 📋 오늘 배울 것

```
STEP 1. EXAONE이란 무엇인가? (API 모델과 무엇이 다른가)
STEP 2. GPU 확인 + 필요한 패키지 설치하기
STEP 3. 토크나이저 & 모델 불러오기
STEP 4. 첫 번째 추론 호출 해보기
STEP 5. 응답 구조 파헤치기
STEP 6. 리소스 사용량 계산해보기
```

---

## 💡 핵심 개념 미리 보기

| 용어 | 한 줄 설명 |
|------|------------|
| **로컬 모델** | 내 GPU에 직접 불러와서 돌리는 모델 (API Key 불필요) |
| **토크나이저** | 텍스트 ↔ 토큰(숫자)을 서로 변환해주는 도구 |
| **ask() 함수** | EXAONE과 대화하기 위해 직접 만드는 호출 창구 (client 역할) |
| **토큰** | 모델이 텍스트를 읽는 최소 단위 (생성 시간의 기준) |
| **finish_reason** | 응답이 왜 끝났는지 알려주는 신호 (직접 계산해서 알아냄) |


---
## STEP 1. EXAONE이란 무엇인가?

### 🏠 "내 집 마련"으로 이해하기

```
API 모델 (GPT 등)    =  호텔 숙박 — 쓸 때마다 요금 지불, 카드키(API Key)로 출입
로컬 모델 (EXAONE)    =  내 집 마련 — 한 번 받아두면 그 다음부터는 계속 무료로 사용
```

### 📌 EXAONE 기본 정보

| 항목 | 내용 |
|------|------|
| 제작사 | LG AI Research |
| 이번에 쓸 모델 | `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct` |
| 받는 곳 | 허깅페이스 허브 (최초 실행 시 자동 다운로드) |
| 필요 조건 | GPU 메모리 약 5~6GB 이상 (Colab 무료 T4로 충분) |

> ⚠️ **API 모델과 가장 큰 차이**  
> API 모델은 호출 1번마다 요금이 발생하지만, 로컬 모델은 GPU에 한 번 올려두면 몇 번을 호출하든 API 요금이 발생하지 않습니다.  
> 대신 **모델을 불러올 만큼의 GPU 메모리**가 반드시 있어야 합니다.


---
## STEP 2. GPU 확인 + 필요한 패키지 설치하기

### 📦 실행 환경 비교

| 방법 | 속도 | 비고 |
|------|------|------|
| CPU로 실행 | ❌ 매우 느림 (사실상 비실용적) | GPU 미할당 시 |
| **GPU로 실행** | ✅ 정상 속도 | Colab에서 T4 GPU 할당 필요 |

### 🔧 GPU 할당 확인 방법 (4단계)

```
① 상단 메뉴 → 런타임 → 런타임 유형 변경 클릭
② 하드웨어 가속기 → GPU 선택
③ GPU 종류 → T4 선택 (무료 등급 기준)
④ 저장 → 런타임 다시 시작 ← 이 단계를 빠뜨리면 GPU가 할당되지 않음!
```


In [1]:
# 🖥️ GPU가 잘 연결되어 있는지 확인하는 코드입니다.
# GPU는 그래픽 카드인데, AI 모델을 빠르게 계산해주는 "엔진" 역할을 합니다.
# (CPU로도 계산이 되긴 하지만 수십~수백 배 느려서 사실상 쓰기 어렵습니다)
#
# 맨 앞의 느낌표(!)는 "이건 파이썬 코드가 아니라 컴퓨터에 직접 내리는 명령어야"라는 뜻입니다.
# nvidia-smi 는 "지금 어떤 GPU가 연결되어 있고, 메모리가 얼마나 있는지" 보여주는 명령어입니다.
#
# ✅ 아래에 GPU 이름(예: Tesla T4)과 메모리 용량이 출력되면 성공!
# ❌ 아무것도 안 뜨거나 에러가 나면 → STEP 2의 4단계(런타임 유형을 GPU로 변경)를 다시 확인하세요.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA GeForce RTX 4090, 24564 MiB


In [13]:
# 📦 이 실습에 필요한 파이썬 라이브러리(도구 모음)를 설치합니다.
#
# transformers : 허깅페이스(HuggingFace)에서 만든 라이브러리로,
#                EXAONE 같은 AI 모델을 "불러오고(load) / 실행(추론)"할 수 있게 해줍니다.
#                모델의 복잡한 내부 코드를 직접 짤 필요 없이 몇 줄이면 됩니다.
# accelerate   : 모델을 GPU 위에 자동으로 잘 올려주는 보조 라이브러리입니다.
#                (아래에서 device_map="auto" 옵션을 쓸 때 내부적으로 이 라이브러리가 동작합니다)
#
# ⚠️ 왜 버전을 "5.2 미만"으로 고정할까요?
#    transformers는 매우 빠르게 업데이트되는 라이브러리입니다. EXAONE 모델을 불러올 때
#    함께 딸려오는 전용 코드(trust_remote_code=True로 실행하는 코드)는 특정 시점의
#    transformers 버전에 맞춰 작성되어 있습니다. 그런데 transformers 5.2 버전부터
#    내부 함수의 인자 이름이 바뀌어서, 최신 버전을 그냥 설치하면
#    "input_embeds" 관련 TypeError가 발생합니다. 그래서 일부러 5.2 이전 버전으로 설치합니다.
#
# !pip install → "파이썬 패키지를 설치해줘"라는 명령어
# -q           → 설치 과정 로그를 자세히 보여주지 않음 (quiet, 조용히)
# --upgrade    → 이미 설치되어 있어도 지정한 조건에 맞는 최신 버전으로 업데이트
!pip install -q "transformers<5.2" --upgrade accelerate

print("✅ 설치 완료!")
print("⚠️ 아래 셀로 넘어가기 전에 상단 메뉴 '런타임 → 런타임 다시 시작'을 꼭 눌러주세요.")
print("   (파이썬은 이미 메모리에 불러온 라이브러리를 기억하고 있어서,")
print("    다시 시작해야만 방금 새로 설치한 버전이 실제로 적용됩니다)")

✅ 설치 완료!
⚠️ 아래 셀로 넘어가기 전에 상단 메뉴 '런타임 → 런타임 다시 시작'을 꼭 눌러주세요.
   (파이썬은 이미 메모리에 불러온 라이브러리를 기억하고 있어서,
    다시 시작해야만 방금 새로 설치한 버전이 실제로 적용됩니다)


---
## STEP 3. 토크나이저 & 모델 불러오기

API 모델의 **"클라이언트"** 에 해당하는 역할을 로컬 모델에서는 **토크나이저 + 모델 + 호출 함수** 가 대신합니다.  
한 번 불러와두면 이후 모든 대화는 이 셋을 통해 이루어집니다.


In [1]:
import os      # 운영체제(컴퓨터) 관련 기능을 쓰기 위한 기본 라이브러리
import torch   # 딥러닝 계산을 담당하는 라이브러리(PyTorch) — GPU 연산도 이걸로 처리합니다

# AutoModelForCausalLM : AI 모델 "본체"를 불러올 때 쓰는 클래스
# AutoTokenizer        : 아래에서 설명할 "토크나이저"를 불러올 때 쓰는 클래스
# 이름 앞에 Auto가 붙은 이유: 모델 이름만 알려주면 어떤 종류의 모델인지 자동으로 맞춰 불러와 줍니다
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 파일은 용량이 커서(수 GB) 다운로드에 시간이 걸릴 수 있습니다.
# 기본 타임아웃(제한 시간)이 짧으면 중간에 끊길 수 있어서 300초(5분)로 넉넉하게 늘려둡니다.
# os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

# 이번에 사용할 모델의 이름입니다. "제작사/모델이름" 형태로 되어 있습니다.
# 허깅페이스 허브(huggingface.co)에 등록된 이름이며,
# 아래 코드가 실행되면 이 이름으로 모델 파일을 자동으로 찾아 다운로드합니다.
model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

print("📥 토크나이저 로드 중...")

# 토크나이저(tokenizer)란?
#   AI 모델은 사실 "글자"를 이해하지 못하고 "숫자"만 이해합니다.
#   토크나이저는 우리가 쓰는 문장을 모델이 이해할 수 있는 숫자(토큰)로 바꿔주고,
#   반대로 모델이 만든 숫자를 다시 사람이 읽는 글자로 바꿔주는 "번역기" 역할을 합니다.
#
# trust_remote_code=True : EXAONE처럼 일부 모델은 transformers 라이브러리에 기본으로
#                           내장되어 있지 않은 자체 코드를 함께 가지고 있습니다.
#                           이 옵션은 "그 코드를 신뢰하고 실행할게"라고 허락해주는 설정입니다.
#                           (공식적으로 배포된 모델이므로 안전하게 사용할 수 있습니다)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("✅ 토크나이저 로드 완료!")

📥 토크나이저 로드 중...


✅ 토크나이저 로드 완료!


In [2]:
# 이제 진짜 AI 모델(두뇌에 해당하는 부분)을 불러옵니다.
# 최초 1회는 인터넷에서 모델 파일을 다운로드하기 때문에 약 2~3분 정도 걸립니다.
# (두 번째 실행부터는 이미 받아둔 파일을 재사용하므로 훨씬 빠릅니다)
print("📥 모델 로드 중... (약 2~3분 소요)")

model = AutoModelForCausalLM.from_pretrained(
    model_name,

    # 숫자를 표현하는 정밀도를 32비트 대신 16비트로 낮춰서 불러옵니다.
    # 정밀도가 살짝 낮아지지만 눈으로 보는 결과에는 차이가 거의 없고,
    # 대신 GPU 메모리를 절반 정도만 사용하게 되어 더 여유 있게 돌릴 수 있습니다.
    torch_dtype=torch.bfloat16,

    # 토크나이저를 불러올 때와 같은 이유로, EXAONE 전용 코드 실행을 허용합니다.
    trust_remote_code=True,

    # 모델을 GPU와 CPU 중 어디에 올릴지 자동으로 결정해줍니다.
    # GPU가 있으면 GPU에, 모자라면 일부를 CPU에 나눠서 올려줍니다.
    device_map="auto",
)

# torch.cuda.memory_allocated() : 지금 GPU에 실제로 올라가 있는 데이터의 용량(byte)을 알려줍니다.
# 1024**3 으로 나누면 byte → GB(기가바이트) 단위로 바꿀 수 있습니다.
print(f"✅ 모델 로드 완료! GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

📥 모델 로드 중... (약 2~3분 소요)


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

✅ 모델 로드 완료! GPU 메모리: 4.5 GB


### ⚠️ 오류가 뜬다면?

| 오류 메시지 | 원인 | 해결 방법 |
|------------|------|----------|
| `OutOfMemoryError` | GPU 메모리 부족 | 런타임 다시 시작 후 재시도 |
| `ReadTimeout` | 다운로드 중 끊김 | 모델 로드 셀만 다시 실행 |
| GPU 이름이 안 뜸 | GPU 미할당 | STEP 2의 4단계를 다시 확인 |
| `create_causal_mask` 관련 `TypeError` | transformers 버전이 너무 최신이라 EXAONE 원격 코드와 안 맞음 | 패키지 설치 셀의 버전 고정(`transformers<5.2`)을 확인하고 **런타임을 다시 시작**한 뒤 처음부터 재실행 |


In [3]:
# GPT의 client 역할을 하는 함수 — 한 번 만들어두면 이후 모든 호출에 재사용
#
# 아래 함수는 조금 길어 보이지만, 하는 일은 딱 하나입니다:
# "messages(대화 내용)를 넣으면 → EXAONE이 생각한 답변을 dict 형태로 돌려준다"
# 한 번 만들어두면 이후에는 이 함수만 호출하면 되고, 내부 과정을 매번 신경 쓸 필요가 없습니다.
def ask(messages, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """EXAONE에게 메시지를 보내고 답변을 반환한다."""

    # [1단계] 대화 내용을 EXAONE 전용 "말투 형식"의 문자열로 바꿉니다.
    #   messages는 [{"role": "user", "content": "..."}] 같은 리스트인데,
    #   모델마다 이걸 하나의 프롬프트 문자열로 조립하는 규칙(chat template)이 정해져 있습니다.
    #   - tokenize=False            : 아직 숫자로 안 바꾸고 "문자열" 그대로 받습니다.
    #   - add_generation_prompt=True: 문자열 맨 끝에 "이제부터 네(모델)가 대답할 차례야"
    #                                 라는 신호를 자동으로 붙여줍니다.
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    # [2단계] 문자열을 모델이 이해하는 "숫자(토큰)"로 바꾸고, GPU로 올립니다.
    #   - return_tensors="pt"      : 결과를 파이토치(PyTorch) 텐서 형태로 받습니다.
    #   - add_special_tokens=False : 1단계의 템플릿이 이미 필요한 특수 표시를 넣었으므로
    #                                여기서 또 추가하면 중복되기 때문에 꺼둡니다.
    #   - .to("cuda")              : 이 데이터를 GPU 메모리로 옮깁니다 ("cuda" = GPU를 가리키는 이름).
    input_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")

    # [3단계] 실제로 모델에게 "다음에 올 말"을 생성시킵니다. (이 부분이 진짜 AI 추론입니다!)
    #   with torch.no_grad(): 는 "학습(훈련)이 아니라 그냥 답만 뽑아낼 거야"라는 뜻입니다.
    #   학습할 때만 필요한 역방향 계산(그래디언트)을 꺼버려서 속도가 빨라지고 메모리도 아낄 수 있습니다.
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,   # 최대 몇 개의 새 토큰까지 생성할지 (= 응답 길이 제한)
            temperature=temperature,          # 답변의 "창의성" 정도. 0에 가까울수록 매번 비슷하고
                                               # 안정적인 답, 높을수록 다양하고 예측하기 어려운 답이 나옵니다.
            top_p=top_p,                      # 다음 단어 후보 중 확률 합이 top_p(예: 0.9 = 90%)가
                                               # 되는 범위 안에서만 고르도록 제한합니다 (엉뚱한 단어 방지).
            do_sample=True if temperature > 0 else False,
                                               # temperature가 0이면 "매번 무조건 1등 후보만 선택"(샘플링 끔),
                                               # 0보다 크면 "확률에 따라 무작위로 선택"(샘플링 켬)합니다.
            eos_token_id=tokenizer.eos_token_id,
                                               # eos = "end of sentence"(문장의 끝). 모델이 이 토큰을
                                               # 생성하면 "할 말 다 했다"는 뜻이므로 생성을 멈춥니다.
        )

    # [4단계] output_ids 안에는 "내가 보낸 질문 + 모델이 만든 답변"이 통째로 들어 있습니다.
    #   그래서 앞부분(내가 보낸 질문 길이만큼)을 잘라내고, 새로 생성된 답변 부분만 남깁니다.
    generated_ids = output_ids[0][input_ids.shape[1]:]

    # 숫자(토큰)로 되어 있는 답변을 다시 사람이 읽는 글자로 바꿉니다.
    # skip_special_tokens=True → 화면에 보일 필요 없는 특수 표시 토큰은 제거하고 출력합니다.
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # [5단계] 답변이 "스스로 할 말을 다 해서" 끝난 건지, 아니면
    #   "max_new_tokens 글자 제한에 걸려서 강제로 잘린" 건지 확인합니다.
    #   → 생성된 마지막 토큰이 eos_token_id와 같으면 "정상 종료", 다르면 "중간에 잘림"으로 판단합니다.
    #   (OpenAI API의 finish_reason과 같은 역할을 우리가 직접 계산해서 만들어주는 것입니다)
    finished_naturally = (
        len(generated_ids) > 0 and generated_ids[-1].item() == tokenizer.eos_token_id
    )

    # 결과를 보기 편하게 딕셔너리(dict) 하나로 묶어서 반환합니다.
    return {
        "content": answer,                                             # 실제 답변 텍스트
        "finish_reason": "stop" if finished_naturally else "length",    # 왜 멈췄는지
        "input_tokens": input_ids.shape[1],                            # 내가 보낸 질문이 토큰 몇 개였는지
        "output_tokens": len(generated_ids),                           # 모델이 답변에 토큰을 몇 개 썼는지
    }

print("✅ ask() 함수 준비 완료! (OpenAI의 client와 같은 역할을 합니다)")

✅ ask() 함수 준비 완료! (OpenAI의 client와 같은 역할을 합니다)


---
## STEP 4. 첫 번째 추론 호출 해보기

이제 EXAONE에게 직접 말을 걸어봅시다!

### 📮 메시지 구조 이해하기 (API 모델과 동일한 방식!)

EXAONE도 GPT와 마찬가지로 `messages` 리스트에 **역할(role)** 과 **내용(content)** 을 담아 보냅니다.

| role | 역할 | 언제 쓰나 |
|------|------|----------|
| `system` | 모델에게 "어떻게 행동할지" 지시 | 맨 처음 한 번 |
| `user` | 나의 질문·요청 | 매번 |
| `assistant` | 모델의 이전 답변 기록 | 대화 이어갈 때 |


In [6]:
# 드디어 EXAONE에게 첫 질문을 보내봅니다!
# 방금 만든 ask() 함수 덕분에, 복잡한 내부 과정을 몰라도
# messages만 잘 채워서 넣으면 답을 받을 수 있습니다.
result = ask(
    messages=[
        # system 메시지: 대화가 시작되기 전에 모델에게 미리 주는 "역할 지시"입니다.
        # 사용자에게는 보이지 않고, 모델이 답변하는 태도에만 영향을 줍니다.
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},

        # user 메시지: 실제로 내가 묻고 싶은 질문입니다.
        {"role": "user",   "content": "파인튜닝이 무엇인지 한 문장으로 설명해 주세요."}
    ],
    temperature=0.7,       # 적당히 자연스러우면서도 다양한 답변이 나오도록 중간 정도 값을 사용
    max_new_tokens=256,    # 답변이 너무 길어지지 않도록 최대 256개의 토큰까지만 생성
)

# result는 dict이므로, 그 중 실제 답변 텍스트인 "content" 값만 꺼내서 출력합니다.
print(result["content"])

파인튜닝은 기존에 학습된 머신러닝 모델의 성능을 특정 작업이나 데이터셋에 더 최적화하기 위해 추가적인 학습 과정을 거치는 기법입니다.


### 🧪 실험: system 프롬프트가 있을 때 vs 없을 때

같은 질문이라도 system 설정에 따라 답변이 완전히 달라집니다.


In [7]:
# 완전히 똑같은 질문("계약 해지 통보는 언제 해야 하나요?")을
# system 메시지만 다르게 해서 두 번 물어보고, 답이 어떻게 달라지는지 비교해봅니다.
# temperature를 0.3으로 낮게 설정해서 "창의성"보다 "일관성"에 초점을 맞춥니다.

# ① system 없이 질문 — user 메시지만 보냅니다 (역할 지시가 전혀 없는 상태)
result_no_sys = ask(
    messages=[
        {"role": "user", "content": "계약 해지 통보는 언제 해야 하나요?"}
    ],
    temperature=0.3,
    max_new_tokens=200,
)

# ② system 있이 질문 — "너는 법률 전문가야, 이렇게 답해"라고 미리 역할을 정해줍니다
result_with_sys = ask(
    messages=[
        {"role": "system", "content": "당신은 대한민국 법률 전문가입니다. 관련 법조항을 인용하여 3문장 이내로 답변하세요."},
        {"role": "user",   "content": "계약 해지 통보는 언제 해야 하나요?"}
    ],
    temperature=0.3,
    max_new_tokens=200,
)

# 두 답변을 나란히 출력해서 눈으로 직접 비교합니다
print("─── system 없음 ───")
print(result_no_sys["content"])

print("\n─── system 있음 ───")
print(result_with_sys["content"])

─── system 없음 ───
계약 해지 통보의 시기는 주로 계약의 종류와 관련 법률에 따라 다르지만, 일반적으로 다음과 같은 원칙과 절차를 따르는 것이 권장됩니다:

1. **계약 조건 확인**: 먼저 해당 계약 문서를 꼼꼼히 검토하여 해지 조건이 명시되어 있는지 확인해야 합니다. 많은 계약에는 해지 조건과 절차가 명시되어 있습니다.

2. **해지 통지 기간 준수**: 대부분의 계약에는 해지 통지 기간이 정해져 있습니다. 이 기간은 보통 해지 통지를 보내야 하는 최소 기간을 의미하며, 이 기간을 지키지 않으면 해지가 무효가 될 수 있습니다. 예를 들어:
   - **단기 계약**: 보통 몇 일에서 몇 주 사이의 짧은 기간이 요구됩니다.
   - **장기 계약**: 몇 달에서 몇 년까지 다양할 수 있으며, 특히 중요한 계약이나 큰 금액이 오가는 경우 더 긴

─── system 있음 ───
계약 해지 통보는 일반적으로 해지 사유가 발생한 날로부터 **14일** 이내에 서면으로 이루어져야 합니다 (**민법 제549조**). 단, 계약 내용에 해지 기간이 명시되어 있다면 그 기간을 준수해야 합니다.


### 🎛️ 주요 파라미터 정리

| 파라미터 | 필수? | 설명 | 예시 |
|---------|-------|------|------|
| `messages` | ✅ | 대화 내용 리스트 | `[{"role":..., "content":...}]` |
| `max_new_tokens` | 선택 | 새로 생성할 최대 토큰 수 | `256` |
| `temperature` | 선택 | 창의성 (0=일정, 높을수록 다양) | `0.7` |
| `top_p` | 선택 | 단어 선택 범위 조절 | `0.9` |
| `do_sample` | 선택 | 확률적으로 뽑을지 여부 | `True` |


---
## STEP 5. 응답 구조 파헤치기

EXAONE의 응답도 단순한 텍스트가 아닙니다. `ask()` 가 **여러 정보를 담은 dict** 로 반환합니다.

```
result (ask()의 반환값)
├── result["content"]         ← 실제 답변 텍스트 ✅ (가장 많이 씀)
├── result["finish_reason"]   ← 왜 생성이 멈췄나? ("stop" / "length")
├── result["input_tokens"]    ← 입력에 사용한 토큰 수
└── result["output_tokens"]   ← 출력에 사용한 토큰 수
```


In [8]:
# 앞에서(첫 번째 호출에서) 받은 result 딕셔너리 안에 어떤 정보가 들어있는지 하나씩 꺼내봅니다.
# result는 {"content": ..., "finish_reason": ..., "input_tokens": ..., "output_tokens": ...} 형태였죠?

print("📝 답변 내용:")
print(result["content"])   # 우리가 가장 많이 쓰게 될, 실제 답변 텍스트

print("\n🏁 종료 이유:", result["finish_reason"])  # "stop"(정상 종료) 또는 "length"(길이 제한으로 잘림)

print("\n📊 토큰 사용량:")
print("   입력 토큰:", result["input_tokens"])    # 내 질문이 토큰 몇 개로 쪼개졌는지
print("   출력 토큰:", result["output_tokens"])   # 모델의 답변이 토큰 몇 개로 이루어졌는지

📝 답변 내용:
파인튜닝은 기존에 학습된 머신러닝 모델의 성능을 특정 작업이나 데이터셋에 더 최적화하기 위해 추가적인 학습 과정을 거치는 기법입니다.

🏁 종료 이유: stop

📊 토큰 사용량:
   입력 토큰: 42
   출력 토큰: 37


### 🚦 finish_reason 이란?

| 값 | 의미 | 대응 방법 |
|----|------|----------|
| `stop` | eos 토큰까지 정상 생성 ✅ | 그대로 사용 |
| `length` | `max_new_tokens` 초과로 잘림 ⚠️ | `max_new_tokens` 값 늘리기 |

> ⚠️ `length` 가 뜨면 답변이 중간에 잘린 겁니다.  
> 특히 JSON 형식 응답이 잘리면 파싱 오류가 납니다!


In [9]:
# 🧪 실험: max_new_tokens 를 너무 작게 하면 어떻게 될까?
# "상세히 설명해주세요"라고 물었는데 답변 길이 제한을 20토큰(짧은 몇 단어 수준)으로
# 아주 짧게 걸어두면 어떤 일이 벌어지는지 직접 확인해봅니다.
result_short = ask(
    messages=[
        {"role": "user", "content": "대한민국 헌법의 기본 원리를 상세히 설명해 주세요."}
    ],
    max_new_tokens=20,   # 의도적으로 아주 짧게 설정 (실제로는 이렇게 짧게 쓰면 안 됩니다!)
)

print("답변:", result_short["content"])              # 문장이 끝맺음도 못하고 뚝 끊길 것입니다
print("종료 이유:", result_short["finish_reason"])    # → "length" 가 나오는지 확인해보세요
print("→ 문장이 중간에 잘렸습니다! max_new_tokens를 늘려야 합니다.")

답변: 대한민국 헌법은 국가의 근본적인 법적 기반이며, 그 안에 내재된 기본
종료 이유: length
→ 문장이 중간에 잘렸습니다! max_new_tokens를 늘려야 합니다.


---
## STEP 6. 리소스 사용량 계산해보기

### 💰 로컬 추론은 정말 무료일까?

GPU에 모델을 올려두고 추론하는 것 자체는 API 요금이 발생하지 않습니다.  
다만 "API로 했다면 얼마였을까?"를 환산해보면, 로컬 모델의 비용 이점을 체감할 수 있습니다.

```
영어: "hello"    → 1 토큰
한국어: "안녕하세요" → 약 3~5 토큰  (한국어가 토큰을 더 씁니다!)
```

### 💵 gpt-4o-mini 단가 (API 환산 비교용, 2025년 기준)

| 구분 | 가격 |
|------|------|
| 입력 토큰 | $0.15 / 100만 토큰 |
| 출력 토큰 | $0.60 / 100만 토큰 |


In [10]:
# 방금 호출한 추론을 "만약 API였다면 얼마였을까?"로 환산해봅니다
# (실제로는 로컬 GPU에서 돌렸기 때문에 이 비용을 내는 것은 아닙니다. 어디까지나 비교용입니다!)

INPUT_PRICE  = 0.15   # gpt-4o-mini 기준: 입력 토큰 100만 개당 $0.15
OUTPUT_PRICE = 0.60   # gpt-4o-mini 기준: 출력 토큰 100만 개당 $0.60
# (참고: 보통 "출력" 토큰 단가가 "입력" 토큰보다 비쌉니다 — 답을 만드는 데 연산이 더 필요하기 때문)

# 앞에서 받은 result 딕셔너리에서 토큰 수를 꺼내옵니다
input_tokens  = result["input_tokens"]
output_tokens = result["output_tokens"]

# 단가 × 사용량 ÷ 100만 = 달러 비용 (100만 토큰당 가격이므로 1_000_000으로 나눕니다)
api_cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000

print(f"입력: {input_tokens}토큰 / 출력: {output_tokens}토큰")
print(f"로컬 추론 비용: ₩0")   # 로컬 GPU에서 실행했으므로 API 요금 자체가 발생하지 않습니다
print(f"(참고) API 환산 비용: ${api_cost:.6f}  (약 {api_cost * 1400:.4f}원)")   # 1달러 ≈ 1,400원으로 환산
print(f"1,000번 호출했다면: 약 {api_cost * 1000 * 1400:.0f}원 절약")   # 같은 질문을 1,000번 했다고 가정했을 때 절약액

입력: 42토큰 / 출력: 37토큰
로컬 추론 비용: ₩0
(참고) API 환산 비용: $0.000029  (약 0.0399원)
1,000번 호출했다면: 약 40원 절약


### 📊 누적 토큰 추적기 만들기

앞으로 EXAONE을 많이 호출할 텐데, 토큰 사용량을 자동으로 합산해주는 도구를 만들어둡니다.  
오늘 만든 `ask()`, `TokenTracker` 는 이후 실습에서 그대로 재사용합니다.


In [11]:
# 호출할 때마다 토큰 사용량을 자동으로 쌓아(누적해)주는 "장부" 같은 도구를 만듭니다.
# class(클래스)는 "이런 기능과 정보를 가진 물건을 찍어내는 틀"이라고 생각하면 됩니다.
class TokenTracker:

    # __init__ : 이 장부를 처음 만들 때(tracker = TokenTracker() 할 때) 딱 한 번 실행되는 부분
    #            여기서 누적값들을 전부 0으로 초기화해둡니다.
    def __init__(self):
        self.total_in  = 0   # 지금까지 사용한 입력 토큰 총합
        self.total_out = 0   # 지금까지 사용한 출력 토큰 총합
        self.calls     = 0   # 지금까지 ask()를 호출한 횟수

    # add() : ask()가 돌려준 result를 넣어주면, 그 안의 토큰 수를 총합에 더해줍니다
    def add(self, result):
        self.total_in  += result["input_tokens"]
        self.total_out += result["output_tokens"]
        self.calls     += 1

    # report() : 지금까지 누적된 사용량과 "API로 했다면" 환산 비용을 보여줍니다
    def report(self):
        api_cost = (self.total_in * 0.15 + self.total_out * 0.60) / 1_000_000
        print(f"총 {self.calls}회 호출 | 입력 {self.total_in} + 출력 {self.total_out} 토큰")
        print(f"로컬 추론 비용: ₩0 | (참고) API 환산 비용: ${api_cost:.4f} (약 {api_cost * 1400:.0f}원)")

# ── 사용법 예시 ──
tracker = TokenTracker()   # 1) 장부를 하나 만들고
tracker.add(result)        # 2) 호출할 때마다 결과를 등록하고
tracker.report()           # 3) 필요할 때 report()로 누적 현황을 확인합니다

총 1회 호출 | 입력 42 + 출력 37 토큰
로컬 추론 비용: ₩0 | (참고) API 환산 비용: $0.0000 (약 0원)


---
## ✅ 오늘 실습 체크리스트

| 항목 | 확인 |
|------|------|
| GPU 확인 완료 (`nvidia-smi` 정상 출력) | ☐ |
| 패키지 설치 완료 (transformers, accelerate) | ☐ |
| 토크나이저 + 모델 로드 성공 (`✅ 모델 로드 완료!`) | ☐ |
| `ask()` 함수로 첫 추론 호출 → 응답 텍스트 출력 성공 | ☐ |
| system 유무에 따른 답변 차이 확인 | ☐ |
| `finish_reason: length` 현상 체험 | ☐ |
| 리소스(토큰) 환산 계산 코드 실행 성공 | ☐ |
| `TokenTracker` 만들고 `report()` 출력 성공 | ☐ |

---

## 🔜 다음 시간 예고

> 다음 시간에는 오늘 만든 `ask()` 와 `TokenTracker` 를 그대로 가져와서,  
> **시스템 프롬프트를 설계**하고 **안전한 재시도 로직** 을 갖춘 **진짜 멀티턴 챗봇** 을 만들어봅니다!
